# GPU integrator benchmark: Yoshida-4 vs Dopri8 (omega Cen spray, rotating-bar host)

Inputs: `bench_inputs.npz` (from `bin/streams/gpu_bench_export.py`) in `My Drive/ocen_gpu/`. Code: `ocen_stream.jaxspray` / `ocen_stream.jaxbench` from GitHub. Results: `My Drive/ocen_gpu/bench_<gpu>/results.json` (+ final states of the full-N runs). Restartable: finished cases are skipped.

Runtime: **Runtime → Change runtime type → A100 GPU**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
!pip install -q --no-deps --force-reinstall "git+https://github.com/vasilybelokurov/ocen_stream@main"
!pip install -q diffrax
!git ls-remote https://github.com/vasilybelokurov/ocen_stream refs/heads/main
import jax, diffrax
print('jax', jax.__version__, 'diffrax', diffrax.__version__, jax.devices())
assert jax.devices()[0].platform == 'gpu', 'no GPU: change the runtime type'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
INP = '/content/drive/MyDrive/ocen_gpu/bench_inputs.npz'
GPU = jax.devices()[0].device_kind.replace(' ', '_')
OUT = f'/content/drive/MyDrive/ocen_gpu/bench_{GPU}'
import glob, hashlib, shutil
if not os.path.exists(INP):                                   # uploaded in < 10 MB parts: rejoin
    with open(INP, 'wb') as fo:
        for f in sorted(glob.glob(INP+'.part_*')):
            with open(f, 'rb') as fi: shutil.copyfileobj(fi, fo)
sha = hashlib.sha256(open(INP, 'rb').read()).hexdigest()
assert sha == 'fbeaa4f69f851924cf76f5b4de0face0263c95d7a95e6565dab16e01ead78b40', sha
print(os.path.getsize(INP)/1e6, 'MB, sha256 ok ->', OUT)

Quick check (2000 particles, both integrators):

In [ ]:
!python -m ocen_stream.jaxbench --inputs "$INP" --out "$OUT/quick" --quick

Full grid: N = 10k, 50k, 179k; Yoshida dt 0.2 / 0.1 Myr, Dopri8 tol 1e-6 / 1e-8 (float64); float32 for Yoshida 0.2 and Dopri8 1e-6 at full N.

In [ ]:
!python -m ocen_stream.jaxbench --inputs "$INP" --out "$OUT"

In [ ]:
import json, pandas as pd
r = json.load(open(f'{OUT}/results.json'))
df = pd.DataFrame([dict(case=k, s=v['seconds'], us_per_particle=v['us_per_particle'], compile_s=v['compile_seconds'],
                        evals_max=v['force_evals_max'], dmu90=v['dmu_pct']['90'], dmu99=v['dmu_pct']['99'], dmu999=v['dmu_pct']['99.9'],
                        agama8_99=v['dmu_agama8_pct']['99'], peak_GB=v['peak_bytes']/1e9, nonfinite=v['nonfinite']) for k, v in r.items()])
pd.set_option('display.width', 200); print(df.to_string(index=False, float_format=lambda x: f'{x:.3g}'))